In [1]:
from pathlib import Path
import os
import re
import json
import gzip
import random
import unicodedata
from collections import Counter

import requests
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
from torch.utils.data import Dataset, DataLoader, random_split

from warcio.archiveiterator import ArchiveIterator
from bs4 import BeautifulSoup
import trafilatura

from langdetect import detect, LangDetectException
import ftfy

from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder

try:
    import lightning.pytorch as pl
except Exception:
    import pytorch_lightning as pl

In [2]:
import emoji
import nltk
import pickle as pk
import string
import time
import torch.nn as nn
import torch.nn.functional as F
import torchvision


from autocorrect import Speller
from functools import lru_cache
from matplotlib import pyplot as plt
from collections import Counter
from nltk import collocations 
from nltk import ngrams
from nltk.corpus import stopwords
from nltk.probability import FreqDist
from nltk.stem.porter import PorterStemmer as ps
from nltk.stem import WordNetLemmatizer as lem
from nltk.tokenize import word_tokenize, wordpunct_tokenize, sent_tokenize
from pymorphy3 import MorphAnalyzer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, \
                                            HashingVectorizer, ENGLISH_STOP_WORDS
from sklearn.feature_selection import RFE, SelectKBest, chi2
from sklearn.linear_model import LogisticRegression 
from sklearn.model_selection import train_test_split
from sklearn.metrics import *
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.utils import resample
from stop_words import get_stop_words
from string import punctuation
from textblob import TextBlob
from tqdm import tqdm

In [3]:
nltk.download('stopwords')
nltk.download('genesis')
nltk.download('punkt')
noise = stopwords.words('russian') + list(punctuation)
punkts = set(string.punctuation)
morph_analyz = MorphAnalyzer()

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\521\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package genesis to
[nltk_data]     C:\Users\521\AppData\Roaming\nltk_data...
[nltk_data]   Package genesis is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\521\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [4]:
PROJECT_ROOT = Path(".").resolve()

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
TOKENIZER_DIR = PROJECT_ROOT / "tokenizers"

for d in [DATA_DIR, RAW_DIR, PROCESSED_DIR, TOKENIZER_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [5]:
devices = [torch.cuda.device(i) for i in range(torch.cuda.device_count())]
for device in devices:
    print(torch.cuda.get_device_name(device))

NVIDIA GeForce RTX 5070 Ti


In [6]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE

'cuda'

### Загрузка

In [ ]:
COMMON_CRAWL_COLLECTION = "CC-MAIN-2024-10"
MAX_WARC_FILES = 1

def download_file(url: str, path: Path, chunk_size: int = 1024 * 1024):
    path.parent.mkdir(parents=True, exist_ok=True)

    if path.exists() and path.stat().st_size > 0:
        print(f"Файл уже существует: {path}")
        return path

    with requests.get(url, stream=True, timeout=60) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))

        with open(path, "wb") as f, tqdm(
            total=total,
            unit="B",
            unit_scale=True,
            desc=path.name
        ) as pbar:
            for chunk in r.iter_content(chunk_size=chunk_size):
                if chunk:
                    f.write(chunk)
                    pbar.update(len(chunk))

    return path

# Функция получает список путей WARC-файлов для выбранного снапшота Common Crawl.
def get_commoncrawl_warc_paths(collection: str):
    url = f"https://data.commoncrawl.org/crawl-data/{collection}/warc.paths.gz"
    response = requests.get(url, timeout=60)
    response.raise_for_status()

    paths = gzip.decompress(response.content).decode("utf-8").splitlines()
    return paths


warc_paths = get_commoncrawl_warc_paths(COMMON_CRAWL_COLLECTION)

print(f"Всего WARC-файлов найдено: {len(warc_paths)}")
print("Первый путь:")
print(warc_paths[0])

In [9]:
downloaded_warc_files = []

for i, rel_path in enumerate(warc_paths[:MAX_WARC_FILES]):
    url = f"https://data.commoncrawl.org/{rel_path}"
    local_path = RAW_DIR / Path(rel_path).name

    print(f"Скачиваем файл {i + 1}/{MAX_WARC_FILES}")
    print(url)

    downloaded_warc_files.append(download_file(url, local_path))

downloaded_warc_files

NameError: name 'warc_paths' is not defined

### Конвертация WARC в текст

In [10]:
# ِФункция извлекает основной текст из HTML.
# Сначала используется trafilatura, затем fallback через BeautifulSoup.
def html_to_text(html: str) -> str | None:
    if not html or not html.strip():
        return None

    extracted = trafilatura.extract(
        html,
        include_comments=False,
        include_tables=False
    )

    if extracted and extracted.strip():
        return extracted.strip()

    soup = BeautifulSoup(html, "xml")

    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()

    text = soup.get_text(separator="\n")
    text = "\n".join(line.strip() for line in text.splitlines() if line.strip())

    return text.strip() if text.strip() else None

# ِФункция конвертирует WARC в jsonl: {"url": ..., "text": ...}
def convert_warc_to_jsonl(warc_path: Path, output_path: Path, max_records: int = 2000):
    output_path.parent.mkdir(parents=True, exist_ok=True)

    saved = 0
    processed = 0

    with gzip.open(warc_path, "rb") as stream, open(output_path, "w", encoding="utf-8") as out:
        for record in tqdm(ArchiveIterator(stream), desc=f"Чтение {warc_path.name}"):
            if record.rec_type != "response":
                continue

            processed += 1

            if processed > max_records:
                break

            http_headers = record.http_headers
            if http_headers is None:
                continue

            content_type = http_headers.get_header("Content-Type") or ""

            if "text/html" not in content_type.lower():
                continue

            url = record.rec_headers.get_header("WARC-Target-URI")

            try:
                raw_bytes = record.content_stream().read()
                html = raw_bytes.decode("utf-8", errors="ignore")
            except Exception:
                continue

            text = html_to_text(html)

            if text:
                item = {
                    "url": url,
                    "text": text
                }
                out.write(json.dumps(item, ensure_ascii=False) + "\n")
                saved += 1

    print(f"Обработано WARC-записей: {processed}")
    print(f"Сохранено текстовых документов: {saved}")

    return output_path

In [11]:
cc_raw_jsonl = RAW_DIR / "commoncrawl_raw_texts.jsonl"

for warc_file in downloaded_warc_files:
    convert_warc_to_jsonl(
        warc_path=warc_file,
        output_path=cc_raw_jsonl,
        max_records=100000
    )

In [12]:
def read_jsonl(path: Path):
    items = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                items.append(json.loads(line))
    return items


cc_raw_items = read_jsonl(cc_raw_jsonl)

print(f"Количество сырых документов: {len(cc_raw_items)}")
print(cc_raw_items[0]["text"][:1000])

Количество сырых документов: 34132
pGzsLw Flash PlayerAЧ Google ChromeBFirefox Microsoft Edge sAiJs][CpGz@wnϥ Flash PlayerAЬ Flash PlayerLkϥθѨMkоCU Chrome sU Firefox sU Edge s
O


### Очистка данных

Выполняется:
- удаление пустых строк;
- Unicode-нормализация;
- нормализация пробелов;
- удаление слишком коротких текстов;
- фильтрация по языку;
- разбиение слишком длинных объектов.

In [13]:
re_url = re.compile('https?://\S+|www\.\S+')
ABC = 'abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ'
ru_ABC ='абвгдеёжзийклмнопрстуфхцчшщъыьэюяАБВГДЕЁЖЗИЙКЛМНОПРСТУФХЦЧШЩЪЫЬЭЮЯ'
denial = ['not', 'cannot', 'doesn',  "wasn't", 'nor', 'couldn', 'weren', 
          "won't", "doesn't", 'needn', "isn't", 'against', "couldn't", 
          'aren', 'isn', 'hadn', 'wouldn', "mightn't", "mustn't", "can't", 
          "wouldn't", 'cant', 'didn', "shouldn't",  "don't", "didn't", 
          "hadn't", "needn't", 'shouldn', 'hasnt', "weren't", 'no', "aren't", 
          "hasn't", "haven't", 'нельзя']
stopwords_english = list(set(get_stop_words("en") + 
                             stopwords.words('english') +
                             list(ENGLISH_STOP_WORDS) +
                             ['user']).difference(denial))
stopwords_russian = list(set(get_stop_words("ru") + 
                             stopwords.words('russian') +
                             ['а','у', 'о', '☕', '♡','♥', 'б', 'аж', 'л', 'ха', 'х'
                              '❤', '❤️','❤️', '❤️', '�','✌️', 'd', '❤',
                              'dd', 'p' 'rt', 'щ', 'з', 'эй', 'ибо', '❄️']).difference(denial))

emoji_pattern = \
re.compile("[" 
           u"\U0001F600-\U0001F64F"  # emoticons
           u"\U0001F300-\U0001F5FF"  # symbols & pictographs
           u"\U0001F680-\U0001F6FF"  # transport & map symbols
           u"\U0001F1E0-\U0001F1FF"  # flags (iOS)
           u"\U00002500-\U00002BEF"  # chinese char
           u"\U0001F191-\U0001F19A"
           u"\U0001F232-\U0001F236" u"\U0001F238-\U0001F23A" u"\U0001F250-\U0001F251"
           u"\U0001F300-\U0001F30C" u"\U0001F30D-\U0001F30E" u"\U0001F313-\U0001F315"
           u"\U0001F316-\U0001F318" u"\U0001F31D-\U0001F31E" u"\U0001F31F-\U0001F320"
           u"\U0001F32D-\U0001F32F" u"\U0001F330-\U0001F331" u"\U0001F332-\U0001F333"
           u"\U0001F334-\U0001F335" u"\U0001F337-\U0001F34A" u"\U0001F34C-\U0001F34F"
           u"\U0001F351-\U0001F37B" u"\U0001F37E-\U0001F37F" u"\U0001F380-\U0001F393"
           u"\U0001F3A0-\U0001F3C4" u"\U0001F3CF-\U0001F3D3" u"\U0001F3E0-\U0001F3E3"
           u"\U0001F3E5-\U0001F3F0" u"\U0001F3F8-\U0001F407" u"\U0001F409-\U0001F40B"
           u"\U0001F40C-\U0001F40E" u"\U0001F40F-\U0001F410" u"\U0001F411-\U0001F412"
           u"\U0001F417-\U0001F429" u"\U0001F42B-\U0001F43E" u"\U0001F442-\U0001F464"
           u"\U0001F466-\U0001F46B" u"\U0001F46C-\U0001F46D" u"\U0001F46E-\U0001F4AC"
           u"\U0001F4AE-\U0001F4B5" u"\U0001F4B6-\U0001F4B7" u"\U0001F4B8-\U0001F4EB"
           u"\U0001F4EC-\U0001F4ED" u"\U0001F4F0-\U0001F4F4" u"\U0001F4F6-\U0001F4F7"
           u"\U0001F4F9-\U0001F4FC" u"\U0001F4FF-\U0001F502" u"\U0001F504-\U0001F507"
           u"\U0001F50A-\U0001F514" u"\U0001F516-\U0001F52B" u"\U0001F52C-\U0001F52D"
           u"\U0001F52E-\U0001F53D" u"\U0001F54B-\U0001F54E" u"\U0001F550-\U0001F55B"
           u"\U0001F55C-\U0001F567" u"\U0001F595-\U0001F596" u"\U0001F5FB-\U0001F5FF"
           u"\U0001F7E0-\U0001F7EB" u"\U0001F90D-\U0001F90F" u"\U0001F910-\U0001F918"
           u"\U0001F919-\U0001F91E" u"\U0001F920-\U0001F927" u"\U0001F928-\U0001F92F"
           u"\U0001F931-\U0001F932" u"\U0001F933-\U0001F93A" u"\U0001F93C-\U0001F93E"
           u"\U0001F940-\U0001F945" u"\U0001F947-\U0001F94B" u"\U0001F94D-\U0001F94F"
           u"\U0001F950-\U0001F95E" u"\U0001F95F-\U0001F96B" u"\U0001F96C-\U0001F970"
           u"\U0001F973-\U0001F976" u"\U0001F977-\U0001F978" u"\U0001F97C-\U0001F97F"
           u"\U0001F980-\U0001F984" u"\U0001F985-\U0001F991" u"\U0001F992-\U0001F997"
           u"\U0001F998-\U0001F9A2" u"\U0001F9A3-\U0001F9A4" u"\U0001F9A5-\U0001F9AA"
           u"\U0001F9AB-\U0001F9AD" u"\U0001F9AE-\U0001F9AF" u"\U0001F9B0-\U0001F9B9"
           u"\U0001F9BA-\U0001F9BF" u"\U0001F9C1-\U0001F9C2" u"\U0001F9C3-\U0001F9CA"
           u"\U0001F9CD-\U0001F9CF" u"\U0001F9D0-\U0001F9E6" u"\U0001F9E7-\U0001F9FF"
           u"\U0001FA70-\U0001FA73" u"\U0001FA78-\U0001FA7A" u"\U0001FA7B-\U0001FA7C"
           u"\U0001FA80-\U0001FA82" u"\U0001FA83-\U0001FA86" u"\U0001FA90-\U0001FA95"
           u"\U0001FA96-\U0001FAA8" u"\U0001FAA9-\U0001FAAC" u"\U0001FAB0-\U0001FAB6"
           u"\U0001FAB7-\U0001FABA" u"\U0001FAC0-\U0001FAC2" u"\U0001FAC3-\U0001FAC5"
           u"\U0001FAD0-\U0001FAD6" u"\U0001FAD7-\U0001FAD9" u"\U0001FAE0-\U0001FAE7"
           u"\U0001FAF0-\U0001FAF6" u"\U00002702-\U000027B0" u"\U00002702-\U000027B0" 
           u"\U000024C2-\U0001F251" u"\U0001f926-\U0001f937" u"\U00010000-\U0010ffff"
           u"\u231A-\u231B" u"\u200d" u"\u23E9-\u23EC" u"\u25FD-\u25FE" u"\u2600-\u2B55" 
           u"\ufe0f"  # dingbats
           u"\u3030" "]+", flags=re.UNICODE)

apostrophe_dict = {
            "ain`t": "am not / are not",
            "aren`t": "are not / am not",
            "can`t": "cannot",
            "can`t`ve": "cannot have",
            "`cause": "because",
            "could`ve": "could have",
            "couldn`t": "could not",
            "couldn`t`ve": "could not have",
            "didn`t": "did not",
            "doesn`t": "does not",
            "don`t": "do not",
            "hadn`t": "had not",
            "hadn`t've": "had not have",
            "hasn`t": "has not",
            "haven`t": "have not",
            "he`d": "he had / he would",
            "he`d`ve": "he would have",
            "he`ll": "he shall / he will",
            "he`ll`ve": "he shall have / he will have",
            "he`s": "he has / he is",
            "how`d": "how did",
            "how`d'y": "how do you",
            "how`ll": "how will",
            "how`s": "how has / how is",
            "i`d": "I had / I would",
            "i`d`ve": "I would have",
            "i`ll": "I shall / I will",
            "i`ll`ve": "I shall have / I will have",
            "i`m": "I am",
            "i`ve": "I have",
            "isn`t": "is not",
            "it`d": "it had / it would",
            "it`d`ve": "it would have",
            "it`ll": "it shall / it will",
            "it`ll`ve": "it shall have / it will have",
            "it`s": "it has / it is",
            "let`s": "let us",
            "ma`am": "madam",
            "mayn`t": "may not",
            "might`ve": "might have",
            "mightn`t": "might not",
            "mightn`t`ve": "might not have",
            "must`ve": "must have",
            "mustn`t": "must not",
            "mustn`t`ve": "must not have",
            "needn`t": "need not",
            "needn`t`ve": "need not have",
            "o`clock": "of the clock",
            "oughtn`t": "ought not",
            "oughtn`t`ve": "ought not have",
            "shan`t": "shall not",
            "sha`n`t": "shall not",
            "shan`t`ve": "shall not have",
            "she`d": "she had / she would",
            "she`d`ve": "she would have",
            "she`ll": "she shall / she will",
            "she`ll`ve": "she shall have / she will have",
            "she`s": "she has / she is",
            "should`ve": "should have",
            "shouldn`t": "should not",
            "shouldn`t`ve": "should not have",
            "so`ve": "so have",
            "so`s": "so as / so is",
            "that`d": "that would / that had",
            "that`d've": "that would have",
            "that`s": "that has / that is",
            "there`d": "there had / there would",
            "there`d`ve": "there would have",
            "there`s": "there has / there is",
            "they`d": "they had / they would",
            "they`d`ve": "they would have",
            "they`ll": "they shall / they will",
            "they`ll`ve": "they shall have / they will have",
            "they`re": "they are",
            "they`ve": "they have",
            "to`ve": "to have",
            "wasn`t": "was not",
            "we`d": "we had / we would",
            "we`d`ve": "we would have",
            "we`ll": "we will",
            "we`ll`ve": "we will have",
            "we`re": "we are",
            "we`ve": "we have",
            "weren`t": "were not",
            "what`ll": "what shall / what will",
            "what`ll`ve": "what shall have / what will have",
            "what`re": "what are",
            "what`s": "what has / what is",
            "what`ve": "what have",
            "when`s": "when has / when is",
            "when`ve": "when have",
            "where`d": "where did",
            "where`s": "where has / where is",
            "where`ve": "where have",
            "who`ll": "who shall / who will",
            "who`ll`ve": "who shall have / who will have",
            "who`s": "who has / who is",
            "who`ve": "who have",
            "why`s": "why has / why is",
            "why`ve": "why have",
            "will`ve": "will have",
            "won`t": "will not",
            "won`t`ve": "will not have",
            "would`ve": "would have",
            "wouldn`t": "would not",
            "wouldn`t`ve": "would not have",
            "y`all": "you all",
            "y`all`d": "you all would",
            "y`all`d`ve": "you all would have",
            "y`all`re": "you all are",
            "y`all`ve": "you all have",
            "you`d": "you had / you would",
            "you`d`ve": "you would have",
            "you`ll": "you shall / you will",
            "you`ll`ve": "you shall have / you will have",
            "you`re": "you are",
            "you`ve": "you have"
        }

@lru_cache(maxsize=2048)
def text_preprocess_light(text):
    text = str(text)
    text = re_url.sub('', text)
    nicks = re.findall(r'@[\w]*', text)
    for nick in nicks:
        text = re.sub(nick, ' ', text)
    re_html = re.compile('<.*?>')
    text = re_html.sub(r'', text)
    text = emoji_pattern.sub(r'',text)
    for word in text.split():
        if word.lower() in apostrophe_dict:
            text = re.sub(word, apostrophe_dict[word.lower()], text)
    text = ' '.join([word for word in text.split() if len(word) > 1])
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    text = re.sub(r'\s\s+', ' ', text)
    text = re.sub('не\s', 'не', text)
    text = re.sub('нет\s', 'нет', text)
    text = ' '.join([word for word in text.split() if len(word) > 2])
    text = text.lower()
#     text = TextBlob(text).correct().string
    text = [morph_analyz.parse(word)[0].normal_form for word in text.split() if word not in stopwords_english] 
                                                                           # if word not in stopwords_russian
    return " ".join(text)

In [14]:
def normalize_unicode(text: str) -> str:
    text = ftfy.fix_text(text)
    text = unicodedata.normalize("NFKC", text)
    return text


def remove_control_chars(text: str) -> str:
    return "".join(
        ch for ch in text
        if ch == "\n" or ch == "\t" or not unicodedata.category(ch).startswith("C")
    )


def normalize_spaces(text: str) -> str:
    text = text.replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    lines = [line.strip() for line in text.splitlines()]
    lines = [line for line in lines if line]

    return "\n".join(lines).strip()


def clean_text_basic(text: str) -> str:
    text = text_preprocess_light(text)
    text = normalize_unicode(text)
    text = remove_control_chars(text)
    text = normalize_spaces(text)
    return text


def detect_language_safe(text: str) -> str | None:
    try:
        sample = text[:2000]
        return detect(sample)
    except LangDetectException:
        return None


def is_allowed_language(text: str, allowed_langs=("en",)) -> bool:
    lang = detect_language_safe(text)
    return lang in allowed_langs


def split_long_text(
    text: str,
    max_words: int = 1024,
    min_words: int = 15
):
    words = text.split()

    if len(words) < min_words:
        return []

    chunks = []

    for start in range(0, len(words), max_words):
        chunk_words = words[start:start + max_words]

        if len(chunk_words) >= min_words:
            chunks.append(" ".join(chunk_words))

    return chunks


def clean_dataset_items(
    items,
    allowed_langs=("en",),
    min_words=15,
    max_words=1024
):
    cleaned = []

    for item in tqdm(items, desc="Очистка текстов"):
        text = item["text"]
        text = clean_text_basic(text)

        if not text:
            continue

        if not is_allowed_language(text, allowed_langs=allowed_langs):
            continue

        chunks = split_long_text(
            text,
            max_words=max_words,
            min_words=min_words
        )

        for chunk in chunks:
            cleaned.append({
                "url": item.get("url"),
                "text": chunk
            })

    return cleaned

In [15]:
cc_clean_items = clean_dataset_items(
    cc_raw_items,
    allowed_langs=("en",),
    min_words=15,
    max_words=1024
)

print(f"После очистки документов: {len(cc_clean_items)}")
print(cc_clean_items[1]["text"][:1000])

Очистка текстов: 100%|██████████████████████████████████████████████████████████| 34132/34132 [04:17<00:00, 132.34it/s]

После очистки документов: 15797
document writeln var mybody document getelementsbytagname body horad var arrhref arrhref reverse var arrimg arrimg reverse var result style padding height eleimg eleimg replace style padding height eleimg eleimg replace style padding height atag innerhtml eleimg mybody insertbefore atag mybody childnodes var divclear document createelement div mybody insertbefore divclear mybody childnodes imgad arrimgsrc arrimghref arrimgtxt var div document createelement div div setattribute style background color black padding update mybody insertbefore div mybody childnodes var odivtxt document getelementbyid divadtxtarea odivtxt parentnode insertbefore div odivtxt var divrow document createelement div divrow setattribute class row divrow setattribute style margin display flex flex wrap wrap justify content space div appendchild divrow divtxt innerhtml txt divimg appendchild divtxt txtad var odivtxt document getelementbyid divadtxtarea arrmenu arrcolor red blueviolet

In [16]:
cc_clean_jsonl = PROCESSED_DIR / "commoncrawl_clean.jsonl"

with open(cc_clean_jsonl, "w", encoding="utf-8") as f:
    for item in cc_clean_items:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

cc_clean_jsonl

WindowsPath('C:/Users/521/CPM Owl/2026/Modern_neural_net_arch/HW1/data/processed/commoncrawl_clean.jsonl')

### Улучшение качества данных

##### Удаление дубликатов

In [17]:
def deduplicate_items(items):
    seen = set()
    unique_items = []

    for item in items:
        text = item["text"]

        if text not in seen:
            unique_items.append(item)
            seen.add(text)

    return unique_items


before = len(cc_clean_items)
cc_unique_items = deduplicate_items(cc_clean_items)
after = len(cc_unique_items)

duplicate_density = before / after if after > 0 else float("inf")

print(f"До удаления дубликатов: {before}")
print(f"После удаления дубликатов: {after}")
print(f"Дублирующая плотность данных: {duplicate_density:.4f}")

До удаления дубликатов: 15797
После удаления дубликатов: 15508
Дублирующая плотность данных: 1.0186


##### Подсчёт энтропии через GPT-2

Для каждого объекта считаем:
$$H(D) = - \sum_i \log p(D_i | D_{<i})$$
Информационная плотность:
$$\rho_{info} = \frac{H(D)}{|D|}$$

In [18]:
gpt2_tokenizer = AutoTokenizer.from_pretrained("openai-community/gpt2")
gpt2_model = AutoModelForCausalLM.from_pretrained("openai-community/gpt2").to(DEVICE)
gpt2_model.eval()

# if gpt2_tokenizer.pad_token is None:
#     gpt2_tokenizer.pad_token = gpt2_tokenizer.eos_token

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [19]:
@torch.no_grad()
def compute_gpt2_entropy(
    text: str,
    tokenizer,
    model,
    device: str = "cpu",
    max_length: int = 1024
):
    encoded = tokenizer(
        text,
        return_tensors="pt",
        add_special_tokens=False
    )

    input_ids = encoded["input_ids"][0]

    if input_ids.numel() < 2:
        return {
            "entropy": np.nan,
            "num_tokens": int(input_ids.numel()),
            "info_density": np.nan
        }

    total_nll = 0.0
    total_tokens = 0

    for start in range(0, input_ids.numel(), max_length):
        chunk = input_ids[start:start + max_length]

        if chunk.numel() < 2:
            continue

        chunk = chunk.unsqueeze(0).to(device)

        outputs = model(
            input_ids=chunk,
            labels=chunk
        )

        # loss — среднее значение negative log likelihood
        # для seq_len - 1 токенов
        n_tokens = chunk.numel() - 1

        total_nll += outputs.loss.item() * n_tokens
        total_tokens += n_tokens

    if total_tokens == 0:
        return {
            "entropy": np.nan,
            "num_tokens": 0,
            "info_density": np.nan
        }

    return {
        "entropy": total_nll,
        "num_tokens": total_tokens,
        "info_density": total_nll / total_tokens
    }

In [20]:
# MAX_TEXTS_FOR_ENTROPY = min(300, len(cc_unique_items))
# cc_entropy_items = cc_unique_items[:MAX_TEXTS_FOR_ENTROPY]
cc_entropy_items = cc_unique_items

entropy_rows = []

for item in tqdm(cc_entropy_items, desc="GPT-2 entropy"):
    stats = compute_gpt2_entropy(
        item["text"],
        tokenizer=gpt2_tokenizer,
        model=gpt2_model,
        device=DEVICE
    )

    entropy_rows.append({
        "url": item.get("url"),
        "text": item["text"],
        **stats
    })

cc_entropy_df = pd.DataFrame(entropy_rows)
cc_entropy_df.head()

GPT-2 entropy: 100%|████████████████████████████████████████████████████████████| 15508/15508 [02:03<00:00, 125.65it/s]


,url,text,entropy,num_tokens,info_density
0,http://080miss.com/V4/?AID=202366&FID=381032&W...,pgzslw flash playera google chromebfirefox edg...,181.794447,29,6.268774
1,http://16ykm.com/vod/detail/id/854829.html,document writeln var mybody document getelemen...,1219.353134,296,4.119436
2,http://199cr.com/archives/25628,not simply traditions option plans wedding dec...,2178.669934,285,7.644456
3,http://20literlife.com/manila-philippines/,major departure authoritative air hard edged b...,1661.939160,214,7.766071
4,http://247sexcams.net/tag/maria/,sex cams fresh porn served day enter username ...,127.979650,15,8.531977


In [21]:
cc_entropy_df.describe()

,entropy,num_tokens,info_density
count,15508.000000,15508.000000,15508.000000
mean,1610.678868,300.707699,6.169453
std,2226.010455,442.035718,1.444793
min,29.070602,14.000000,0.115105
25%,334.881640,52.000000,5.433437
50%,744.530060,126.000000,6.370183
75%,1822.862657,326.000000,7.116990
max,24810.679787,4204.000000,12.381447


###### Информационная плотность всего датасета
Используем взвешенное среднее:
$$\rho_{info}(D) = \frac{\sum_i H(D_i)}{\sum_i |D_i|}$$

In [22]:
dataset_info_density = (
    cc_entropy_df["entropy"].sum()
    / cc_entropy_df["num_tokens"].sum()
)

print(f"Информационная плотность Common Crawl: {dataset_info_density:.4f}")

Информационная плотность Common Crawl: 5.3563


##### Фильтрация объектов с очень низкой и очень высокой энтропией

In [23]:
def filter_by_entropy_quantiles(
    df: pd.DataFrame,
    low_q: float = 0.05,
    high_q: float = 0.95
):
    df = df.dropna(subset=["info_density"]).copy()

    low = df["info_density"].quantile(low_q)
    high = df["info_density"].quantile(high_q)

    filtered = df[
        (df["info_density"] >= low)
        & (df["info_density"] <= high)
    ].copy()

    print(f"Нижняя граница: {low:.4f}")
    print(f"Верхняя граница: {high:.4f}")
    print(f"До фильтрации: {len(df)}")
    print(f"После фильтрации: {len(filtered)}")

    return filtered


cc_filtered_df = filter_by_entropy_quantiles(
    cc_entropy_df,
    low_q=0.05,
    high_q=0.95
)

cc_filtered_df.head()

Нижняя граница: 3.4553
Верхняя граница: 8.1917
До фильтрации: 15508
После фильтрации: 13956


,url,text,entropy,num_tokens,info_density
0,http://080miss.com/V4/?AID=202366&FID=381032&W...,pgzslw flash playera google chromebfirefox edg...,181.794447,29,6.268774
1,http://16ykm.com/vod/detail/id/854829.html,document writeln var mybody document getelemen...,1219.353134,296,4.119436
2,http://199cr.com/archives/25628,not simply traditions option plans wedding dec...,2178.669934,285,7.644456
3,http://20literlife.com/manila-philippines/,major departure authoritative air hard edged b...,1661.939160,214,7.766071
7,http://3eapg.sussvil.com/tuition-and-financial...,utica university receives national recognition...,357.113647,50,7.142273


In [24]:
cc_final_texts = cc_filtered_df["text"].tolist()

print(f"Финальное количество текстов Common Crawl: {len(cc_final_texts)}")
print(cc_final_texts[3][:1000])

Финальное количество текстов Common Crawl: 13956
major departure authoritative air hard edged beauty singapore eventful manila suits trevor wedding follow post remarkable charms rough exterior instance stayed hotel hostel quarter mile ocean called buoy owner domineering asshole fortunately met staff hand delight female staff attached restaurant bar sweet led favor hotel constant drinking days sit drink buckets beer stupid heard play pool hang buoy girls spent lot time girls pleased report reconnected college pal cat prettier tolentino family hails manila happened vacation met family names relationships forgot learned cat recommended boracay crazier experiences trip post forthcoming spent time hanging american military guy named jack living cebu absolutely insane lucky told stories laughing days afterward jack reading real pleasure hope road sentimental stuff distinctions note manila worst food vince joked filipino national dish unadorned boiled hotdog plate rice burger cat excellent ex

### Токенизация текста

##### Символьная токенизация

In [25]:
class CharTokenizer:
    def __init__(self, special_tokens=None):
        if special_tokens is None:
            special_tokens = ["<PAD>", "<UNK>"]

        self.special_tokens = special_tokens
        self.token_to_id = {}
        self.id_to_token = {}

    def fit(self, texts):
        chars = sorted(set("".join(texts)))
        vocab = self.special_tokens + chars

        self.token_to_id = {tok: i for i, tok in enumerate(vocab)}
        self.id_to_token = {i: tok for tok, i in self.token_to_id.items()}

    def encode(self, text):
        unk_id = self.token_to_id["<UNK>"]
        return [
            self.token_to_id.get(ch, unk_id)
            for ch in text
        ]

    def decode(self, ids):
        return "".join(
            self.id_to_token.get(i, "<UNK>")
            for i in ids
        )

    @property
    def vocab_size(self):
        return len(self.token_to_id)

In [26]:
char_tokenizer = CharTokenizer()
char_tokenizer.fit(cc_final_texts)

sample_text = random.choice(cc_final_texts)
char_ids = char_tokenizer.encode(sample_text)

print(f"Размер словаря символьного токенизатора: {char_tokenizer.vocab_size}")
print(f"Длина случайного объекта в символах-токенах: {len(char_ids)}")
print(sample_text[:500])

Размер словаря символьного токенизатора: 29
Длина случайного объекта в символах-токенах: 123
nearest store music films books prices reviews music film features events offers stores careers fopp faq feb fopplist cover


##### Токенизация по словам

In [27]:
class WordTokenizer:
    def __init__(self, special_tokens=None, max_vocab_size=None):
        if special_tokens is None:
            special_tokens = ["<PAD>", "<UNK>"]

        self.special_tokens = special_tokens
        self.max_vocab_size = max_vocab_size
        self.token_to_id = {}
        self.id_to_token = {}

    def _tokenize(self, text):
        return re.findall(r"\w+|[^\w\s]", text.lower(), flags=re.UNICODE)

    def fit(self, texts):
        counter = Counter()

        for text in texts:
            counter.update(self._tokenize(text))

        most_common = counter.most_common(self.max_vocab_size)

        vocab_words = [word for word, _ in most_common]
        vocab = self.special_tokens + vocab_words

        self.token_to_id = {tok: i for i, tok in enumerate(vocab)}
        self.id_to_token = {i: tok for tok, i in self.token_to_id.items()}

    def encode(self, text):
        unk_id = self.token_to_id["<UNK>"]
        return [
            self.token_to_id.get(tok, unk_id)
            for tok in self._tokenize(text)
        ]

    def decode(self, ids):
        return " ".join(
            self.id_to_token.get(i, "<UNK>")
            for i in ids
        )

    @property
    def vocab_size(self):
        return len(self.token_to_id)

In [29]:
word_tokenizer = WordTokenizer(max_vocab_size=8000)
word_tokenizer.fit(cc_final_texts)

sample_text = random.choice(cc_final_texts)
word_ids = word_tokenizer.encode(sample_text)

print(f"Размер словаря word-токенизатора: {word_tokenizer.vocab_size}")
print(f"Длина случайного объекта в word-токенах: {len(word_ids)}")
print(sample_text[:500])

Размер словаря word-токенизатора: 8002
Длина случайного объекта в word-токенах: 18
worship sunday bible study sunday wednesday play not applicable singing singings delivered app device subscribe favorite podcast player


##### BPE-токенизация

In [30]:
BPE_VOCAB_SIZE = 8000

def train_bpe_tokenizer(
    texts,
    vocab_size: int = 8000,
    save_path: Path | None = None
):
    tokenizer = Tokenizer(BPE(unk_token="<UNK>"))

    tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=False)
    tokenizer.decoder = ByteLevelDecoder()

    trainer = BpeTrainer(
        vocab_size=vocab_size,
        special_tokens=["<PAD>", "<UNK>", "<BOS>", "<EOS>"]
    )

    tokenizer.train_from_iterator(
        texts,
        trainer=trainer,
        length=len(texts)
    )

    if save_path is not None:
        save_path.parent.mkdir(parents=True, exist_ok=True)
        tokenizer.save(str(save_path))

    return tokenizer

In [31]:
bpe_tokenizer_path = TOKENIZER_DIR / "commoncrawl_bpe.json"

bpe_tokenizer = train_bpe_tokenizer(
    cc_final_texts,
    vocab_size=BPE_VOCAB_SIZE,
    save_path=bpe_tokenizer_path
)

sample_text = random.choice(cc_final_texts)
bpe_ids = bpe_tokenizer.encode(sample_text).ids

print(f"Размер словаря BPE-токенизатора: {bpe_tokenizer.get_vocab_size()}")
print(f"Длина случайного объекта в BPE-токенах: {len(bpe_ids)}")
print(sample_text[:500])

Размер словаря BPE-токенизатора: 8000
Длина случайного объекта в BPE-токенах: 310
product info care model usb port hub expandable usb portsmaximum mbps data transmission speed supportequipped brightness ledmulti tap designedplay playsaves power individual switchcompatible smartphone smart tap keyboard mouse storing device box terabyte port usb speed super hub mbps gadget bucket terabyte hub black white specifications products products brand paisawapas cash coupons ensures price deal gadget bucket terabyte hub black white gadget bucket india ensure cheapest price gadget bucket


### Датасет WikiText
Теперь выполняем аналогичную обработку для wikitext.

In [32]:
wiki_dataset = load_dataset("wikitext", "wikitext-2-raw-v1")
wiki_dataset

DatasetDict({
    test: Dataset({
        features: ['text'],
        num_rows: 4358
    })
    train: Dataset({
        features: ['text'],
        num_rows: 36718
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 3760
    })
})

In [33]:
wiki_raw_items = []

for split in ["train", "validation", "test"]:
    for item in wiki_dataset[split]:
        text = item["text"]

        if text and text.strip():
            wiki_raw_items.append({
                "url": f"wikitext/{split}",
                "text": text
            })

print(f"Количество сырых объектов WikiText: {len(wiki_raw_items)}")
print(wiki_raw_items[2]["text"])

Количество сырых объектов WikiText: 29119
 The game began development in 2010 , carrying over a large portion of the work done on Valkyria Chronicles II . While it retained the standard features of the series , it also underwent multiple adjustments , such as making the game more forgiving for series newcomers . Character designer Raita Honjou and composer Hitoshi Sakimoto both returned from previous entries , along with Valkyria Chronicles II director Takeshi Ozawa . A large team of writers handled the script . The game 's opening theme was sung by May 'n . 



##### Очистка WikiText

In [34]:
wiki_clean_items = clean_dataset_items(
    wiki_raw_items,
    allowed_langs=("en",),
    min_words=10,
    max_words=700
)

print(f"После очистки WikiText: {len(wiki_clean_items)}")
print(wiki_clean_items[0]["text"][:1000])

Очистка текстов: 100%|██████████████████████████████████████████████████████████| 29119/29119 [01:47<00:00, 269.63it/s]

После очистки WikiText: 18178
senj valkyria unrecorded chronicles japanese lit valkyria battlefield commonly referred valkyria chronicles iii japan tactical role playing video game developed sega media vision playstation portable released january japan game valkyria series employing fusion tactical real time gameplay predecessors story runs parallel game nameless penal military unit serving nation gallia europan war perform secret black operations pitted against imperial unit calamaty raven


##### Удаление дубликатов WikiText

In [35]:
wiki_unique_items = deduplicate_items(wiki_clean_items)

print(f"До удаления дубликатов: {len(wiki_clean_items)}")
print(f"После удаления дубликатов: {len(wiki_unique_items)}")

До удаления дубликатов: 18178
После удаления дубликатов: 18173


##### Энтропия WikiText через GPT-2

In [36]:
# MAX_WIKI_TEXTS_FOR_ENTROPY = min(300, len(wiki_unique_items))
# wiki_entropy_items = wiki_unique_items[:MAX_WIKI_TEXTS_FOR_ENTROPY]
wiki_entropy_items = wiki_unique_items
wiki_entropy_rows = []

for item in tqdm(wiki_entropy_items, desc="WikiText GPT-2 entropy"):
    stats = compute_gpt2_entropy(
        item["text"],
        tokenizer=gpt2_tokenizer,
        model=gpt2_model,
        device=DEVICE
    )

    wiki_entropy_rows.append({
        "url": item.get("url"),
        "text": item["text"],
        **stats
    })

wiki_entropy_df = pd.DataFrame(wiki_entropy_rows)
wiki_entropy_df.head()

WikiText GPT-2 entropy: 100%|███████████████████████████████████████████████████| 18173/18173 [01:23<00:00, 217.75it/s]


,url,text,entropy,num_tokens,info_density
0,wikitext/train,senj valkyria unrecorded chronicles japanese l...,542.368835,84,6.456772
1,wikitext/train,game development carrying portion valkyria chr...,390.574835,52,7.511055
2,wikitext/train,met positive sales japan praised japanese west...,352.150764,59,5.968657
3,wikitext/train,previous valkyira chronicles games valkyria ch...,693.439888,104,6.667691
4,wikitext/train,game battle blitz carried valkyira chronicles ...,866.613836,123,7.045641


In [37]:
wiki_info_density = (
    wiki_entropy_df["entropy"].sum()
    / wiki_entropy_df["num_tokens"].sum()
)

print(f"Информационная плотность WikiText: {wiki_info_density:.4f}")

Информационная плотность WikiText: 6.8033


In [38]:
wiki_filtered_df = filter_by_entropy_quantiles(
    wiki_entropy_df,
    low_q=0.05,
    high_q=0.95
)

wiki_final_texts = wiki_filtered_df["text"].tolist()

print(f"Финальное количество текстов WikiText: {len(wiki_final_texts)}")

Нижняя граница: 5.6737
Верхняя граница: 8.5989
До фильтрации: 18173
После фильтрации: 16355
Финальное количество текстов WikiText: 16355


##### Токенизация WikiText

###### Символьная токенизация

In [39]:
wiki_char_tokenizer = CharTokenizer()
wiki_char_tokenizer.fit(wiki_final_texts)

sample_wiki_text = random.choice(wiki_final_texts)
wiki_char_ids = wiki_char_tokenizer.encode(sample_wiki_text)

print(f"WikiText char vocab size: {wiki_char_tokenizer.vocab_size}")
print(f"Длина случайного объекта: {len(wiki_char_ids)}")

WikiText char vocab size: 29
Длина случайного объекта: 457


###### Word-токенизация

In [40]:
wiki_word_tokenizer = WordTokenizer(max_vocab_size=8000)
wiki_word_tokenizer.fit(wiki_final_texts)

sample_wiki_text = random.choice(wiki_final_texts)
wiki_word_ids = wiki_word_tokenizer.encode(sample_wiki_text)

print(f"WikiText word vocab size: {wiki_word_tokenizer.vocab_size}")
print(f"Длина случайного объекта: {len(wiki_word_ids)}")

WikiText word vocab size: 8002
Длина случайного объекта: 36


###### BPE-токенизация WikiText

Используем BPE-токенизатор, обученный на Common Crawl.

In [41]:
sample_wiki_text = random.choice(wiki_final_texts)
wiki_bpe_ids = bpe_tokenizer.encode(sample_wiki_text).ids

print(f"BPE vocab size: {bpe_tokenizer.get_vocab_size()}")
print(f"Длина случайного WikiText объекта в BPE-токенах: {len(wiki_bpe_ids)}")

BPE vocab size: 8000
Длина случайного WikiText объекта в BPE-токенах: 119


### Packed batching

Packed batching объединяет несколько коротких последовательностей в один объект фиксированной длины.

Например, если block_size = 512, то несколько коротких текстов могут быть склеены в один блок длиной 512 токенов.

Маска:
 
    0 — <PAD>;
    1 — токены первого объекта;
    2 — токены второго объекта;
    3 — токены третьего объекта и так далее.


In [42]:
PAD_TOKEN = "<PAD>"
PAD_ID = bpe_tokenizer.token_to_id(PAD_TOKEN)
PAD_ID

0

In [43]:
def encode_texts_bpe(texts, tokenizer):
    sequences = []

    for text in tqdm(texts, desc="BPE encode"):
        ids = tokenizer.encode(text).ids

        if len(ids) > 0:
            sequences.append(ids)

    return sequences


wiki_bpe_sequences = encode_texts_bpe(
    wiki_final_texts,
    bpe_tokenizer
)

print(f"Количество BPE-последовательностей WikiText: {len(wiki_bpe_sequences)}")
print(f"Пример длины: {len(wiki_bpe_sequences[0])}")

BPE encode: 100%|█████████████████████████████████████████████████████████████| 16355/16355 [00:01<00:00, 11833.81it/s]

Количество BPE-последовательностей WikiText: 16355
Пример длины: 96


In [44]:
def pack_sequences(
    sequences,
    block_size: int,
    pad_id: int
):
    packed_input_ids = []
    packed_masks = []

    current_ids = []
    current_mask = []
    current_object_id = 1

    def flush():
        nonlocal current_ids, current_mask, current_object_id

        if len(current_ids) == 0:
            return

        pad_len = block_size - len(current_ids)

        current_ids = current_ids + [pad_id] * pad_len
        current_mask = current_mask + [0] * pad_len

        packed_input_ids.append(current_ids)
        packed_masks.append(current_mask)

        current_ids = []
        current_mask = []
        current_object_id = 1

    for seq in sequences:
        # Если последовательность длиннее block_size,
        # режем её на куски.
        chunks = [
            seq[i:i + block_size]
            for i in range(0, len(seq), block_size)
        ]

        for chunk in chunks:
            if len(chunk) == 0:
                continue

            # Если текущий блок переполнится, сохраняем его.
            if len(current_ids) + len(chunk) > block_size:
                flush()

            current_ids.extend(chunk)
            current_mask.extend([current_object_id] * len(chunk))
            current_object_id += 1

            # Если блок полностью заполнен, сохраняем.
            if len(current_ids) == block_size:
                flush()

    flush()

    return (
        torch.tensor(packed_input_ids, dtype=torch.long),
        torch.tensor(packed_masks, dtype=torch.long)
    )

In [45]:
BLOCK_SIZE = 512

packed_ids, packed_masks = pack_sequences(
    wiki_bpe_sequences,
    block_size=BLOCK_SIZE,
    pad_id=PAD_ID
)

print(f"Packed input ids shape: {packed_ids.shape}")
print(f"Packed masks shape: {packed_masks.shape}")

Packed input ids shape: torch.Size([3249, 512])
Packed masks shape: torch.Size([3249, 512])


In [46]:
print("Пример input_ids:")
print(packed_ids[0][:100])

print("\nПример packed mask:")
print(packed_masks[0][:100])

Пример input_ids:
tensor([7350,   13,  705, 2744, 4354,  151, 6268,   46, 4471,  294, 3245, 6608,
         705, 2744, 4354, 3981, 2486, 5926, 6786,  705, 2744, 4354, 4471,  294,
        4313, 1624, 5112,  788, 2064, 2866,  787,  911, 2711,  137, 2176, 1208,
        2679,  435,   95,   81, 6890, 3642, 2037, 1624,  911,  705, 2744, 4354,
        1221,  748,   47, 6000, 5112,  788,  628,  264,  426,  282, 5513,  184,
           7, 5652,  238, 1551, 5133, 5713,  911, 5345, 4112, 5908, 3810, 2019,
        4684, 3015, 1946,  179, 1424,  718,   41,  993,  707, 3285,  767, 2981,
        5265,  557, 1506, 5991,  180, 2019,  656,   83,   51,   28,  107, 6613,
          10, 1388,  924, 3053])

Пример packed mask:
tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1

##### PyTorch Dataset для packed batching

In [47]:
class PackedLanguageModelingDataset(Dataset):
    def __init__(
        self,
        input_ids: torch.Tensor,
        packed_masks: torch.Tensor,
        pad_id: int
    ):
        self.input_ids = input_ids
        self.packed_masks = packed_masks
        self.pad_id = pad_id

    def __len__(self):
        return self.input_ids.shape[0]

    def __getitem__(self, idx):
        input_ids = self.input_ids[idx]
        packed_mask = self.packed_masks[idx]

        labels = input_ids.clone()

        # PAD-токены не должны участвовать в loss
        labels[input_ids == self.pad_id] = -100

        return {
            "input_ids": input_ids,
            "packed_mask": packed_mask,
            "labels": labels
        }

In [48]:
packed_dataset = PackedLanguageModelingDataset(
    input_ids=packed_ids,
    packed_masks=packed_masks,
    pad_id=PAD_ID
)

sample = packed_dataset[0]

print(sample.keys())
print(sample["input_ids"].shape)
print(sample["packed_mask"].shape)
print(sample["labels"].shape)

dict_keys(['input_ids', 'packed_mask', 'labels'])
torch.Size([512])
torch.Size([512])
torch.Size([512])


##### LightningDataModule

In [49]:
class PackedWikiTextDataModule(pl.LightningDataModule):
    def __init__(
        self,
        dataset: Dataset,
        batch_size: int = 8,
        val_ratio: float = 0.1,
        num_workers: int = 0
    ):
        super().__init__()

        self.dataset = dataset
        self.batch_size = batch_size
        self.val_ratio = val_ratio
        self.num_workers = num_workers

    def setup(self, stage=None):
        val_size = int(len(self.dataset) * self.val_ratio)
        train_size = len(self.dataset) - val_size

        self.train_dataset, self.val_dataset = random_split(
            self.dataset,
            [train_size, val_size],
            generator=torch.Generator().manual_seed(SEED)
        )

    def train_dataloader(self):
        return DataLoader(
            self.train_dataset,
            batch_size=self.batch_size,
            shuffle=True,
            num_workers=self.num_workers
        )

    def val_dataloader(self):
        return DataLoader(
            self.val_dataset,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=self.num_workers
        )

In [50]:
data_module = PackedWikiTextDataModule(
    dataset=packed_dataset,
    batch_size=4,
    val_ratio=0.1,
    num_workers=0
)

data_module.setup()

batch = next(iter(data_module.train_dataloader()))

print(batch["input_ids"].shape)
print(batch["packed_mask"].shape)
print(batch["labels"].shape)

torch.Size([4, 512])
torch.Size([4, 512])
torch.Size([4, 512])


### Итоговые результаты

In [51]:
summary = {
    "Common Crawl raw documents": len(cc_raw_items),
    "Common Crawl cleaned documents": len(cc_clean_items),
    "Common Crawl unique documents": len(cc_unique_items),
    "Common Crawl final documents": len(cc_final_texts),
    "Common Crawl duplicate density": duplicate_density,
    "Common Crawl info density": dataset_info_density,
    "Char vocab size": char_tokenizer.vocab_size,
    "Word vocab size": word_tokenizer.vocab_size,
    "BPE vocab size": bpe_tokenizer.get_vocab_size(),
    "WikiText raw documents": len(wiki_raw_items),
    "WikiText cleaned documents": len(wiki_clean_items),
    "WikiText unique documents": len(wiki_unique_items),
    "WikiText final documents": len(wiki_final_texts),
    "WikiText info density": wiki_info_density,
    "Packed dataset size": len(packed_dataset),
    "Packed block size": BLOCK_SIZE
}

pd.DataFrame([summary]).T.rename(columns={0: "value"})

,value
Common Crawl raw documents,34132.000000
Common Crawl cleaned documents,15797.000000
Common Crawl unique documents,15508.000000
Common Crawl final documents,13956.000000
Common Crawl duplicate density,1.018636
Common Crawl info density,5.356294
Char vocab size,29.000000
Word vocab size,8002.000000
BPE vocab size,8000.000000
WikiText raw documents,29119.000000


В ходе лабораторной работы были выполнены следующие пункты:
- скачан WARC-файл Common Crawl;
- HTML-страницы были извлечены из WARC и преобразованы в текст;
- выполнена первичная очистка:
  - удаление HTML;
  - удаление пустых строк;
  - нормализация Unicode;
  - нормализация пробелов;
  - фильтрация по языку;
  - разбиение длинных текстов.
- удалены дубликаты и рассчитана дублирующая плотность данных;
- с помощью GPT-2 рассчитаны:
   - энтропия объектов;
   - информационная плотность объектов;
   - информационная плотность всего датасета.
- удалены объекты с экстремально низкой и высокой информационной плотностью;
- реализованы:
   - символьная токенизация;
   - word-токенизация;
   - BPE-токенизация.
- загружен и обработан датасет WikiText;
- для WikiText выполнена токенизация с использованием BPE, обученного на Common Crawl.
- реализован механизм packed batching.
- реализован LightningDataModule для получения батчей фиксированной длины.